In [ ]:
from sqlalchemy import create_engine
import urllib.parse

from faker import Faker
fake = Faker('en_US')
Faker.seed(42)

import numpy as np
np.seterr(divide = 'ignore', invalid='ignore')

import pandas as pd
pd.options.display.max_columns = None
pd.set_option('display.max_rows', 100)

import matplotlib
matplotlib.use('Agg')
from matplotlib import rcParams
%matplotlib inline
rcParams['figure.figsize'] = 19, 11

import os, random

from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

cwd = os.getcwd()
os.makedirs('./Output', exist_ok=True) 
os.makedirs('./Input', exist_ok=True)  
os.makedirs('./model_save/model', exist_ok=True) 

fake_sk = Faker("sk_SK")
fake_us = Faker("en_US")
fakers = [fake_sk, fake_us]

Faker.seed(42)
random.seed(42)
np.random.seed(42)

address = './Input/ibm_hr_synthetic_data.csv'
df = pd.read_csv(address)

print("Original shape:", df.shape)

def generate_identity_for_gender(gender: str):
    faker = random.choice(fakers)

    # First name
    if gender == 'Male':
        first = faker.first_name_male()
    elif gender == 'Female':
        first = faker.first_name_female()
    else:
        first = faker.first_name()

    # Last name logic (SK needs gender!)
    if faker == fake_sk:
        if gender == 'Male':
            last = faker.last_name_male()
        elif gender == 'Female':
            last = faker.last_name_female()
        else:
            last = faker.last_name()
    else:
        # US priezviská sú vždy gender-neutral
        last = faker.last_name()

    full = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}@sk_company.com"
    username = f"{first[0].lower()}{last.lower()}"

    return pd.Series(
        [first, last, full, email, username],
        index=['FirstName', 'LastName', 'FullName', 'Email', 'Username']
    )

df[['FirstName', 'LastName', 'FullName', 'Email', 'Username']] = df['Gender'].apply(generate_identity_for_gender)

cat_cols = [
            'Attrition',
            'Over18',
            'BusinessTravel',
            'Gender',
            'EducationField',
            'EnvironmentSatisfaction',
            'JobInvolvement',
            'JobLevel',
            'JobRole',
            'JobSatisfaction',
            'PerformanceRating',
            'RelationshipSatisfaction',
            'WorkLifeBalance',
            'MaritalStatus',
            'OverTime',
            'Department',
            'DistanceFromHome',
            'Education',
            'FirstName', 
            'LastName', 
            'FullName', 
            'Email', 
            'Username'
            ]

cat_distributions = {
    col: df[col].value_counts(normalize=True)
    for col in cat_cols
}

numeric_cols = df.select_dtypes(include=['int64']).columns.tolist()

n_new = len(df) * 5
synthetic_rows = []

for _ in range(n_new):
    row = {}
    for col in cat_cols:
        vals = cat_distributions[col].index
        probs = cat_distributions[col].values
        row[col] = np.random.choice(vals, p=probs)
    for col in numeric_cols:
        row[col] = int(np.random.choice(df[col].values))

    gender = row.get('Gender', None)
    faker = random.choice(fakers)

    if gender == 'Male':
        first = faker.first_name_male()
    elif gender == 'Female':
        first = faker.first_name_female()
    else:
        first = faker.first_name()

    last = faker.last_name()
    full = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}@sk_company.com"
    username = f"{first[0].lower()}{last.lower()}"

    row['FirstName'] = first
    row['LastName'] = last
    row['FullName'] = full
    row['Email'] = email
    row['Username'] = username

    synthetic_rows.append(row)

df_new = pd.DataFrame(synthetic_rows)
df_new = df_new.reindex(columns=df.columns)
df_extended = pd.concat([df, df_new], ignore_index=True)


if 'EmployeeNumber' in df_extended.columns:
    df_extended['EmployeeNumber'] = range(1, len(df_extended) + 1)

for c in cat_cols:
    if c in df_extended.columns:
        df_extended[c] = df_extended[c].astype('category')

print("Extended shape:", df_extended.shape)

display(df_extended)


In [ ]:
df_extended.info()

In [ ]:
### Save to SQL Server database table
server   = '192.168.50.85,1433'
database = 'data_science'
username = 'sa'
password = 'W4rpDr1v3@'

driver = 'ODBC Driver 17 for SQL Server'

driver_encoded = urllib.parse.quote_plus(driver)
password_encoded = urllib.parse.quote_plus(password)

connection_string = (
    f'mssql+pyodbc://{username}:{password_encoded}@{server}/{database}'
    f'?driver={driver_encoded}&Encrypt=no&TrustServerCertificate=yes'
)

print('Connecting with:', connection_string)

engine = create_engine(connection_string)
table_name = 'HR_Synth_Data'
df_extended.to_sql(table_name, con=engine, if_exists='append', index=False, schema='dbo')
#df.info()